In [ ]:
# 파이썬 유틸리티 함수 정의
import matplotlib.pyplot as plt
import numpy as np
from skimage.io import imread
from skimage.color import rgb2gray, rgba2rgb, gray2rgb

def plot_image(image, title=''):
    plt.imshow(image)
    plt.axis('off')
    plt.title(title, size=20)

In [ ]:
from scipy import fftpack

im = np.mean(imread("parrot.png"), axis=2) / 255

imnoisy = im.copy()
for n in range(imnoisy.shape[1]):      # x 축 코사인 노이즈 추가
  imnoisy[:, n] += np.cos(0.1*np.pi*n)

imgfft = fftpack.fft2((im).astype(float))
imgfreq = fftpack.fftshift( imgfft )

noisyfft = fftpack.fft2((imnoisy).astype(float))
noisyfreq = fftpack.fftshift( noisyfft )


plt.figure(figsize=(15,10))
plt.gray()

plt.subplot(2,2,1), plot_image(im, 'Original Image')
plt.subplot(2,2,2), plot_image( (20*np.log10( 0.1 + imgfreq)).astype(int), 'Original Spectrum')

plt.subplot(2,2,3), plot_image(imnoisy, 'Sinusoidal Noisy Image')
plt.subplot(2,2,4), plot_image( (20*np.log10( 0.1 + noisyfreq)).astype(int), 'Noisy Spectrum')


plt.show()

In [ ]:
noisyfreq[170:176,:220] = noisyfreq[170:176,231:] = 0  # eliminate the frequencies most

recover = fftpack.ifft2(fftpack.ifftshift( noisyfreq )).real

plt.gray()
plt.figure(figsize=(10,10))
plt.subplot(1,2,1), plot_image( (20*np.log10( 0.1 + noisyfreq)).astype(int), 'Noisy Spectrum')
plt.subplot(1,2,2), plot_image(recover, 'Recovered Image')
plt.show()

In [ ]:
from skimage.transform import hough_line, hough_line_peaks, hough_circle, hough_circle_peaks
from skimage.draw import circle_perimeter
import cv2

image = cv2.imread('triangle_circle.png', cv2.IMREAD_GRAYSCALE)
image = image / 255.0

# Classic straight-line Hough transform
h, theta, d = hough_line(image)
_, theta, d = hough_line_peaks(h, theta, d, num_peaks=10)

radrng = np.arange(50, 100, 2)
#radrng = np.arange(50, 200, 2)
hough_res = hough_circle(image, radrng)
accums, cx, cy, radii = hough_circle_peaks(hough_res, radrng, total_num_peaks=6)

image_cir = gray2rgb(image)
for center_y, center_x, radius in zip(cy, cx, radii):
    circy, circx = circle_perimeter(center_y, center_x, radius)
    image_cir[circy, circx] = (0.9, 0.2, 0.2)

plt.figure(figsize=(10, 10))

plt.subplot(221)
plot_image(image, 'Source')

plt.subplot(222)
plt.title('Hough Transform')
plt.imshow(h, extent=[10*np.rad2deg(theta[-1]), np.rad2deg(theta[0]), d[-1], d[0]],
           cmap='hot', aspect=1/1.5)

ax = plt.subplot(223)
ax.imshow(image, cmap='gray')
for angle, dist in zip(theta, d):
    y0 = (dist - 0 * np.cos(angle)) / np.sin(angle)
    y1 = (dist - image.shape[1] * np.cos(angle)) / np.sin(angle)
    ax.plot((0, image.shape[1]), (y0, y1), '-r')
ax.set_xlim((0, image.shape[1]))
ax.set_ylim((image.shape[0], 0))
ax.set_axis_off()
ax.set_title('Detected lines', size=20)

plt.subplot(224)
plot_image(image_cir, 'Detected Circles')

plt.show()